# 01｜建立併購事件樣本（v5）

把 TEJ 下載的 MOPS 重大訊息，整理成研究用的事件表 `events.csv`。

**流程**
1. 檢查資料：確認 Drive 上的檔案都在，看每種檔案的欄位
2. 合併 MOPS 公告並檢查完整性
3. 第一輪篩選：依公告的表格類型挑出候選公告，預填欄位
4. 第二輪篩選：確認股價、估計相對規模、補強標記、把同一宗交易的公告歸成一組
5. 產生審閱用的檔案：審閱表（Excel）和全文閱讀頁（HTML）
6. （下一版）讀回審閱結果，檢查並產生 `events.csv`

**篩選條件**
- 納入：公告當時是上市櫃的主併公司、取得控制權、相對規模 ≥ 1%、首次公告
- 排除：少數股權、集團內部重組、純資產購買、金融業
- 標記：[-5,+5] 內有其他重大訊息、盤後公告、關係人交易

**資料**（TEJ，放在 Google Drive 的 `ma_event_study/data/`）
- `tej_DAILY_2018～2025`：個股日資料（報酬率、市值、成交量），含市場報酬指數 Y9997
- `tej_company_info`：公司基本資料（產業別、上市別、下市日期）
- `tej_mops_all_2018～2023`：MOPS 重大訊息，含主旨和全文，包含非交易日的公告（2018/1/1～2023/6/30）
- `tej_financials`：季財務資料

## 第 1 格｜檢查資料

連接 Google Drive，列出 `data` 資料夾裡的檔案，每一種檔案各打開一個，顯示欄位名稱和前 3 列。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time
import pandas as pd
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', 250)

DATA_DIR = '/content/drive/MyDrive/ma_event_study/data'
CACHE_DIR = DATA_DIR.replace('/data', '/cache')      # 整理好的資料存在這裡，下次直接讀，不用重新轉檔
os.makedirs(CACHE_DIR, exist_ok=True)

files = sorted(f for f in os.listdir(DATA_DIR) if f.lower().endswith('.xlsx'))
print(f'資料夾裡共有 {len(files)} 個 Excel 檔：')
for f in files:
    print(f'  {f:45s} {os.path.getsize(os.path.join(DATA_DIR, f)) / 1e6:7.1f} MB')

def peek(keyword, label=''):
    """找出檔名包含 keyword 的第一個檔案，顯示欄位名稱和前 3 列"""
    hits = [f for f in files if keyword in f.lower()]
    print('\n' + '=' * 90)
    if not hits:
        print(f'【{label}】找不到檔名包含「{keyword}」的檔案')
        return
    print(f'【{label}】{hits[0]}（這一類共 {len(hits)} 個檔案）')
    df = pd.read_excel(os.path.join(DATA_DIR, hits[0]), nrows=5, dtype=str)
    print('欄位名稱：', df.columns.tolist())
    display(df.head(3))

peek('daily', '個股日資料')
peek('company', '公司基本資料')
peek('mops_all', 'MOPS 重大訊息')
peek('financ', '財務資料')

## 第 2 格｜合併 MOPS 公告，並檢查完整性

1. 把 6 個年度檔案合併成一張表 `mops`，每則公告有主旨和全文
2. 清理格式：日期轉成日期格式；`_x000D_`（Excel 的換行符號）換成真正的換行
3. **檢查 1**：每個檔案讀到的筆數，和下載時 TEJ 畫面上顯示的總筆數比對
4. **檢查 2**：公告的星期分布。公告任何一天都可能發布，所以星期六、日也應該有資料

> 資料完整性紀錄：第一次下載時沒有勾選 TEJ 的「日曆日」，資料只涵蓋交易日。檢查星期分布時發現星期日完全沒有公告、每年有公告的天數剛好等於交易日數，重新下載後補回 9,142 則非交易日的公告（約 2.5%）。

第一次執行要幾分鐘，之後只要幾秒。

In [ ]:
!pip -q install python-calamine

RENAME = {'代號': 'stock_id', '名稱': 'firm_name', '發言日期': 'ann_date', '序號': 'seq',
          '發言時間': 'ann_time', '法條': 'law', '款': 'clause', '事實發生日': 'fact_date',
          '主旨': 'subject', '內容': 'content'}

def read_xlsx(path):
    try:
        return pd.read_excel(path, dtype=str, engine='calamine')   # 比較快的讀法
    except Exception:
        return pd.read_excel(path, dtype=str)                      # 讀不了就用一般的讀法

TEJ_COUNT = {2018: 61197, 2019: 62859, 2020: 66807, 2021: 76735, 2022: 70744, 2023: 40541}   # 在 TEJ 畫面上看到的總筆數

def load_mops_all():
    cache = f'{CACHE_DIR}/mops_all.pkl'
    if os.path.exists(cache):
        print('讀取已整理好的檔案')
        return pd.read_pickle(cache)
    names = [f for f in files if 'mops_all' in f.lower()]
    print(f'找到 {len(names)} 個檔案（應該是 6 個）')
    parts = []
    for f in names:
        t = time.time()
        df = read_xlsx(os.path.join(DATA_DIR, f))
        df.columns = [str(c).strip() for c in df.columns]
        df['source_file'] = f
        parts.append(df)
        print(f'  {f}: {len(df):,} 筆（{time.time() - t:.0f} 秒）')
    df = pd.concat(parts, ignore_index=True).rename(columns=RENAME)
    df['ann_date'] = pd.to_datetime(df['ann_date'], errors='coerce')
    df['fact_date'] = pd.to_datetime(df['fact_date'], errors='coerce')
    for c in ['subject', 'content']:
        df[c] = df[c].fillna('').str.replace('_x000D_', '\n', regex=False).str.strip()
    if len(names) == 6:                              # 六個檔案都到齊才存起來
        df.to_pickle(cache)
    return df

mops = load_mops_all()

# ── 檢查 1：每個檔案的筆數，和 TEJ 畫面上的總筆數是否一致 ──
chk = mops.groupby('source_file').agg(筆數=('stock_id', 'size'), 最早日期=('ann_date', 'min'),
                                      最晚日期=('ann_date', 'max'), 沒有全文=('content', lambda s: (s == '').sum()))
chk['TEJ筆數'] = [TEJ_COUNT.get(int(d.year)) for d in chk['最早日期']]
chk['一致'] = ['✓' if a == b else '✗ 不一致' for a, b in zip(chk['筆數'], chk['TEJ筆數'])]
print('\n【檢查 1】各檔案筆數'); display(chk)

# ── 檢查 2：星期分布（這次應該有星期六、日）──
wk = ['一', '二', '三', '四', '五', '六', '日']
by_day = mops.groupby(mops['ann_date'].dt.dayofweek).size()
by_day.index = ['星期' + wk[i] for i in by_day.index]
print('\n【檢查 2】各星期的公告筆數'); print(by_day.to_string())
print('\n每年有公告的天數：'); print(mops.groupby(mops['ann_date'].dt.year)['ann_date'].nunique().to_string())
print(f'\n合計 {len(mops):,} 筆公告')

## 第 3 格｜篩出候選的併購公告，並預填欄位

重大訊息的全文是照固定表格填寫的。這一格依**表格類型**篩選，比只用主旨關鍵字準確：

| 表格類型 | 第一個欄位 | 內容 |
|---|---|---|
| 併購 | 併購種類 | 董事會決議合併、收購、股份轉換 |
| 股份交換 | 股份交換之標的公司名稱 | 發行新股交換他公司股份 |
| 公開收購申報 | 公開收購申報日期 | 本公司對他公司發動公開收購 |
| 取得有價證券 | 標的物之名稱及性質 | 取得他公司股權（累積持股超過 50% 才列入） |
| 其他 | 事實發生日 等 | 自由格式，改用主旨關鍵字篩選 |

程式另外做三件事：
1. **預填欄位**：從全文讀出併購種類、標的公司、是否關係人、對價種類、持股比例、金額線索
2. **自動標記**：程式看得出來的排除理由（集團內部重組、本公司是被併方、後續公告、金融業……）。標記只是建議，最後由人判斷
3. **列出同一家公司前一年內更早的候選公告**，幫助判斷是不是首次公告

> 探索階段的發現：主旨關鍵字「換股」會誤中「更換股務代理機構」；「合併」會誤中各種合併財務報表；同一個「款」裡混著很多不相干的公告，所以不用款來篩選（上市、上櫃的併購表格都在第 11 款，興櫃在第 9 款；這個差異在第 4 格用來辨認興櫃公司）。

In [ ]:
import re, unicodedata
import numpy as np

WORK_DIR = DATA_DIR.replace('/data', '/work')
os.makedirs(WORK_DIR, exist_ok=True)
nfkc = lambda s: unicodedata.normalize('NFKC', s) if isinstance(s, str) else ''

# ══ 0) 文字清理：主旨接成一行；全文的換行統一；把特殊字元（全形、部首字）轉成一般字 ══
mops['subject'] = mops['subject'].fillna('').str.replace(r'[\r\n]+', '', regex=True).map(nfkc)
mops['content'] = mops['content'].fillna('').str.replace('\r', '\n', regex=False)
head = mops['content'].str[:80].map(nfkc).str.replace(r'\s+', '', regex=True)

# ══ 1) 解析工具：把條列式的公告全文拆成一項一項 ══
def split_items(text):
    # 只有編號剛好是「上一項 + 1」才算新的一項，避免把內文裡的小標號當成新欄位
    items, cur_no, buf = [], 0, []
    for line in str(text).split('\n'):
        m = re.match(r'\s*(\d{1,2})\s*[.．、]\s*(.*)', line)
        if m and int(m.group(1)) == cur_no + 1:
            if cur_no: items.append((cur_no, ''.join(buf)))
            cur_no, buf = cur_no + 1, [m.group(2)]
        else:
            buf.append(line.strip())
    if cur_no: items.append((cur_no, ''.join(buf)))
    out = []
    for no, body in items:
        m = re.search(r':', body)
        label, value = (body[:m.start()], body[m.end():]) if m else (body[:30], '')
        out.append((no, label.strip(), value.strip()))
    return out

def field(items, *keys):
    for key in keys:
        for _, label, value in items:
            if key in label: return value
    return ''

def template_of(items):
    first = items[0][1] if items else ''
    if '併購種類' in first:                    return '併購'
    if '股份交換之標的公司' in first:           return '股份交換'
    if '公開收購申報日期' in first:             return '公開收購申報'
    if '接獲公開收購人' in first:               return '被收購方'
    if '標的物之名稱及性質' in first and '特別股' in first: return '取得有價證券'
    if '證券名稱' in first:                     return '買賣有價證券'
    if '標的物之名稱及性質' in first:           return '取得資產'
    return '其他'

def yes_no(s):
    s = str(s).strip()
    if s.startswith('是'): return '是'
    if s.startswith(('否', '不適用', '非', '無')): return '否'
    return s[:6]

def guess_payment(s):
    s = str(s)
    cash  = bool(re.search(r'現金|自有資金|借款|融資', s))
    stock = bool(re.search(r'發行新股|增資發行|新股|普通股為對價|股份為對價|股票為對價|換發', s))
    if cash and stock: return 'mixed?'
    if cash:  return 'cash?'
    if stock: return 'stock?'
    return ''

PCT = re.compile(r'(\d{1,3}(?:\.\d+)?)\s*%')
def max_pct(s):
    s = str(s)
    vals = [float(x) for x in PCT.findall(s) if float(x) <= 100]
    if re.search(r'百分之百|全部股份|全數股', s): vals.append(100.0)
    return max(vals) if vals else None

AMT = re.compile(r'(?:總金額|總價金|總對價|總價款|交易金額|收購價金|現金對價總計|對價總計|合計)[^。;\n]{0,50}?[\d,\.]+\s*(?:億|萬|仟|千|百萬)?\s*(?:元|歐元|美元|美金|圓)')
def amount_hints(text, n=3):
    out = []
    for m in AMT.finditer(str(text).replace('\n', '')):
        h = m.group(0)[:70]
        if h not in out: out.append(h)
    return ' | '.join(out[:n])

def parse_row(subject, content):
    content = nfkc(content)
    it = split_items(content)
    tpl = template_of(it)
    r = dict(template=tpl, deal_type='', target_raw='', counterparty='', related='', relation='',
             consideration='', ratio='', pct_after=None, amount_hint='', plan='')
    if tpl == '併購':
        r.update(deal_type=field(it, '併購種類')[:20], target_raw=field(it, '參與併購公司名稱')[:150],
                 counterparty=field(it, '交易相對人(')[:150], related=yes_no(field(it, '交易相對人為關係人')),
                 relation=field(it, '交易相對人與公司之關係')[:150], consideration=field(it, '對價種類')[:150],
                 ratio=field(it, '換股比例')[:150], plan=field(it, '併購完成後之計畫')[:250],
                 amount_hint=amount_hints(content))
        r['pct_after'] = max_pct(r['relation'] + r['plan'] + field(it, '後續處理方式'))
    elif tpl == '股份交換':
        r.update(deal_type='股份交換', target_raw=field(it, '標的公司名稱')[:150], counterparty=field(it, '交易相對人')[:150],
                 related=yes_no(field(it, '交易相對人為關係人')), relation=field(it, '交易相對人與公司之關係')[:150],
                 consideration='發行新股(股份交換)', ratio=field(it, '換股比例')[:150],
                 pct_after=max_pct(field(it, '累積持有')), amount_hint=field(it, '累積持有')[:120])
    elif tpl == '公開收購申報':
        r.update(deal_type='公開收購', target_raw=field(it, '被收購有價證券之公開發行公司名稱')[:150],
                 counterparty=field(it, '公開收購人之公司名稱')[:150], consideration=field(it, '對價種類')[:150],
                 ratio=('數量:' + field(it, '被收購之有價證券數量')[:60] + ' / 條件:' + field(it, '如達到預定收購數量')[:120]),
                 amount_hint=(field(it, '預定收購之有價證券價格')[:40] + ' | ' + amount_hints(content)),
                 plan=field(it, '併購完成後之計畫')[:250])
    elif tpl in ('取得有價證券', '買賣有價證券'):
        r.update(deal_type='取得有價證券', target_raw=field(it, '標的物之名稱', '證券名稱')[:150],
                 counterparty=field(it, '交易相對人及其與公司之關係')[:150], related=yes_no(field(it, '為關係人交易')),
                 consideration=field(it, '資金來源')[:150], pct_after=max_pct(field(it, '累積持有')),
                 amount_hint=field(it, '交易總金額')[:150])
    else:
        r.update(amount_hint=amount_hints(content), pct_after=max_pct(subject))
    r['payment_guess'] = guess_payment(r['consideration'])
    return r

# ══ 2) 先挑出「值得解析」的公告 ══
EQUITY   = r'股權|普通股|股份(?!有限)|股票'
KW_FREE  = (r'簡易合併|吸收合併|合併案|合併契約|合併事宜|進行合併|決議.{0,25}合併|收購|併購|股份轉換|股份交換'
            r'|取得.{0,40}(?:' + EQUITY + r')|受讓.{0,30}(?:股份|股權)')
EX_FREE  = (r'股務代理|財務報|財報|XBRL|營收|營運|自結|帳齡|比率|高流動|債權人|庫藏股|買回|處理程序|減資|公司債|金融債|債券'
            r'|理財|基金|設備|標案|恩慈|特別委員會|併購委員會|審議委員會|接獲|應賣|期間屆滿|成就|專戶|第33條|43條之1|369條'
            r'|收購本公司|受託銀行|分公司|澄清|媒體|報導|記者會|法人說明會|法說會|土地|不動產|使用權')
S = mops['subject']
is_ma_form  = head.str.contains('^1[.．]?(?:併購種類|股份交換之標的公司|公開收購申報日期)')
is_sec_form = head.str.contains('^1[.．]?(?:標的物之名稱及性質|證券名稱)') & S.str.contains('取得') & S.str.contains(EQUITY)
is_free     = ~head.str.contains('^1[.．]?(?:併購種類|股份交換之標的公司|公開收購申報日期|標的物之名稱及性質|證券名稱|接獲公開收購人)') \
              & S.str.contains(KW_FREE) & ~S.str.contains(EX_FREE)
n_target_side = int(head.str.contains('^1[.．]?接獲公開收購人').sum())
pre = mops[is_ma_form | is_sec_form | is_free].copy()
print(f'全部公告 {len(mops):,} 則 → 初步挑出 {len(pre):,} 則'
      f'（併購類表格 {int(is_ma_form.sum()):,}、取得股權表格 {int(is_sec_form.sum()):,}、其他格式但主旨含關鍵字 {int(is_free.sum()):,}）')
print(f'另有 {n_target_side:,} 則是「本公司被別人公開收購」的公告，不列入')

# ══ 3) 解析全文，預填欄位 ══
parsed = pd.DataFrame([parse_row(s, c) for s, c in zip(pre['subject'], pre['content'])], index=pre.index)
cand = pd.concat([pre, parsed], axis=1)

# 「取得股權」表格：累積持股比例看得出不超過 50% 的，是少數股權，直接不列入
minority = cand['template'].isin(['取得有價證券', '買賣有價證券']) & (cand['pct_after'].fillna(101) <= 50)
unknown_market = (cand['template'] == '買賣有價證券') & cand['pct_after'].isna()
print(f'取得股權表格中，累積持股 ≤ 50%（少數股權）：{int(minority.sum()):,} 則，不列入；'
      f'市場買賣且看不出持股比例：{int(unknown_market.sum()):,} 則，不列入')
cand = cand[~minority & ~unknown_market & (cand['template'] != '取得資產')].copy()

# ══ 4) 公司基本資料：產業、上市別 ══
info = pd.read_excel(f'{DATA_DIR}/tej_company_info.xlsx', dtype=str)
info.columns = [str(c).strip() for c in info.columns]
info = info.rename(columns={'代號': 'stock_id', '上市別': 'listing', 'TSE產業_名稱': 'tse_ind', 'TEJ子產業_名稱': 'tej_ind',
                            '下市日期': 'delist_date'})[['stock_id', 'listing', 'tse_ind', 'tej_ind', 'delist_date']]
info['stock_id'] = info['stock_id'].str.strip()
cand = cand.merge(info.drop_duplicates('stock_id'), on='stock_id', how='left')
FIN = r'金融|銀行|保險|證券|金控|票券|期貨|壽險|產險'
cand['is_fin'] = cand['tse_ind'].fillna('').str.contains(FIN) | cand['tej_ind'].fillna('').str.contains(FIN)

# ══ 5) 自動標記：程式看得出來的排除理由（只是建議，最後由人判斷）══
def flags(r):
    f, s = [], r['subject']
    who = s + ' ' + r['target_raw'] + ' ' + r['relation'] + ' ' + r['counterparty']
    if r['is_fin']:                                                   f.append('金融業')
    if '分割' in r['deal_type'] and not re.search('合併|收購', r['deal_type']): f.append('分割')
    if re.search(r'簡易合併|[二兩]家子公司|孫公司|組織重整|組織架構|集團架構', who) or \
       (r['related'] == '是' and re.search(r'子公司|孫公司', who)):
        f.append('集團內部重組?')
    elif r['related'] == '是':                                        f.append('關係人交易')
    if re.search(r'終止上市|終止上櫃|終止興櫃|停止公開發行|撤銷公開發行|解散', s) or \
       re.search(r'本公司(?:將|擬|將會|即)?(?:成為.{0,40}子公司|終止上市|終止上櫃|下市|下櫃|解散|消滅)', s + r['plan']) or \
       re.search(r'(?:被收購公司|讓與方|消滅公司)[:：][^:：]{0,45}本公司', r['target_raw']):
        f.append('本公司是被併方?')
    if re.search(r'補充|更正|基準日|變更|核准|屆滿|延長|進度|完成|終止|撤銷|股東會|修正|說明|時程|更新|狀況', s):
        f.append('後續公告?')
    if r['template'] in ('取得有價證券', '買賣有價證券'):
        if re.search(r'債|基金|理財|受益|特別股|票券|存單', r['target_raw']): f.append('非普通股')
        if re.search(r'增資|新設|設立|認購', s + r['target_raw']):            f.append('增資或新設')
        if '處分' in s.replace('取得或處分資產', ''):                  f.append('含處分')
    if r['template'] == '其他' and pd.notna(r['pct_after']) and r['pct_after'] <= 50:
        f.append('少數股權?')
    return '、'.join(f)

cand['auto_flag'] = cand.apply(flags, axis=1)
STRONG = r'金融業|分割|集團內部重組|本公司是被併方|非普通股|增資或新設|含處分|少數股權'
cand['priority'] = np.where(cand['auto_flag'] == '', 'A 優先審閱',
                   np.where(cand['auto_flag'].str.contains(STRONG), 'C 建議排除', 'B 需要確認'))
cand['after_hours'] = (cand['ann_time'].fillna('') > '13:30:00').map({True: '盤後', False: '盤中'})

# ══ 6) 同一家公司前一年內有沒有更早的候選公告（幫忙判斷是不是首次公告）══
cand = cand.sort_values(['stock_id', 'ann_date', 'ann_time']).reset_index(drop=True)
prev_n, prev_s = [], []
for sid, g in cand.groupby('stock_id', sort=False):
    dates, subs = g['ann_date'].tolist(), g['subject'].tolist()
    for i, dt in enumerate(dates):
        js = [j for j in range(i) if (dt - dates[j]).days <= 365]
        prev_n.append(len(js))
        prev_s.append(f"{dates[js[-1]].date()} {subs[js[-1]][:40]}" if js else '')
cand['n_prev_1y'], cand['prev_ann'] = prev_n, prev_s

# ══ 7) 統計與存檔 ══
rev = cand[cand['ann_date'] >= '2019-01-01'].copy()
print(f'\n候選公告（2019/1/1 之後）：{len(rev):,} 則，來自 {rev["stock_id"].nunique():,} 家公司')
print('\n【表格類型 × 優先順序】'); display(pd.crosstab(rev['template'], rev['priority'], margins=True))
print('\n【年份 × 優先順序】');     display(pd.crosstab(rev['ann_date'].dt.year, rev['priority'], margins=True))
print('\n【自動標記的理由】（一則公告可能有多個）')
display(rev['auto_flag'].str.split('、').explode().replace('', '（無）').value_counts().to_frame('則數').T)
print('\n【被標為金融業的產業名稱】', sorted(set(rev.loc[rev['is_fin'], 'tej_ind'].dropna())))
print('【上市別】', rev['listing'].fillna('（對不到）').value_counts().to_dict())

cand.to_pickle(f'{WORK_DIR}/candidates_v1.pkl')
ILLEGAL = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f]')
cols = ['priority', 'auto_flag', 'stock_id', 'firm_name', 'ann_date', 'ann_time', 'after_hours', 'clause', 'template', 'subject',
        'deal_type', 'target_raw', 'counterparty', 'related', 'relation', 'consideration', 'payment_guess', 'ratio',
        'pct_after', 'amount_hint', 'plan', 'n_prev_1y', 'prev_ann', 'listing', 'tse_ind', 'tej_ind']
out = rev[cols].copy()
for c in out.columns:
    out[c] = out[c].map(lambda v: ILLEGAL.sub('', v) if isinstance(v, str) else v)
out_path = f'{WORK_DIR}/candidates_v1.xlsx'
out.to_excel(out_path, index=False)
print(f'\n已存出：{out_path}')

## 第 4 格｜第二輪篩選：確認股價、歸組成「宗」、估計相對規模

第 3 格篩出的候選公告有 2,639 則（含 2018 年，用來比對首次公告），其中很多是同一宗交易的重複公告。這一格做五件事：

1. **確認公告當時有股價**：讀個股日資料，計算每則公告的事件日 t0（非交易日或盤後公告順延到下一個交易日）、估計期 [-150,-31] 的有效天數（成交量為 0 的日子不算）。有效天數不到 100 天的不能進樣本
2. **估計相對規模**：從表格的金額欄位讀出交易金額，除以宣告前一個月底的市值。外幣用近似匯率換算，只供篩選參考，正式的金額由人工確認
3. **補強自動標記**（v5 修改）
   - 公告當時是不是興櫃公司：TEJ 的日資料包含興櫃股票，所以「有股價」不代表已經上市櫃。興櫃公司的公告依據另一套規章，款的編號不同，用這個來辨認
   - 取得股權的表格：用「本次買進股數」和「交易後累積股數」算出交易**前**的持股比例。交易前已經超過 50% 的，控制權沒有改變，建議排除
   - 集團內部重組分成兩種：證據明確的（主旨或欄位寫明標的是本公司的子公司）建議排除；只知道交易對象是關係人的，改列「需要確認」，由人判斷
   - 公開收購預定收購的比例不到 50% 的，標記出來提醒確認有沒有取得控制權
4. **把同一宗交易的多則公告歸成一組**：同一家公司一年內的公告，如果提到相同的標的名稱，就歸為同一宗（v5 新增：標的欄位一字不差的也算同一宗）。每一宗以最早的公告日為首次公告日
5. **列出同一家公司前後一年內的其他候選交易**，審閱時用來檢查有沒有被拆開的同一宗交易

輸出分兩層：
- **第一層**：這一宗至少有一則制式表格公告（併購、公開收購申報、股份交換、取得有價證券）
- **第二層**：只有自由格式的公告

> 檢查紀錄：v4 的結果（第一層 A 154、B 60、C 686 宗）逐筆檢查後發現四個問題：(1) 約 10 宗是興櫃公司；(2) 有的交易仍被拆成多宗（例如環球晶收購 Siltronic 被拆成 3 宗）；(3) 「集團內部重組」的規則太寬，把向關係人收購的真實併購也排除了；(4) 也有太鬆的地方，例如亞泥買嘉惠電力，交易前已持股 59.7%。v5 針對這四點修改。

第一次執行要讀 8 個日資料檔，約 2～3 分鐘；之後直接讀整理好的檔案。

In [ ]:
# ══ 1) 讀個股日資料（第一次約 2～3 分鐘，之後直接讀整理好的檔案）══
def load_daily():
    cache = f'{CACHE_DIR}/daily.pkl'
    if os.path.exists(cache):
        print('讀取已整理好的日資料')
        return pd.read_pickle(cache)
    parts = []
    for f in [f for f in files if 'daily' in f.lower()]:
        t = time.time()
        df = read_xlsx(os.path.join(DATA_DIR, f))
        df.columns = [str(c).strip() for c in df.columns]
        ren = {}
        for c in df.columns:                           # 欄位名稱用關鍵字對應，避免全形、半形符號不同
            for k, v in {'代號': 'stock_id', '年月日': 'date', '報酬率': 'ret', '市值': 'mktcap', '成交量': 'vol'}.items():
                if k in c: ren[c] = v
        df = df.rename(columns=ren)
        df['stock_id'] = df['stock_id'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        for c in ['ret', 'mktcap', 'vol']:
            df[c] = pd.to_numeric(df[c], errors='coerce')
        parts.append(df[['stock_id', 'date', 'ret', 'mktcap', 'vol']])
        print(f'  {f}: {len(df):,} 筆（{time.time() - t:.0f} 秒）')
    daily = pd.concat(parts, ignore_index=True).dropna(subset=['date'])
    daily.to_pickle(cache)
    return daily

daily = load_daily()
print(f'日資料共 {len(daily):,} 筆，{daily["stock_id"].nunique():,} 檔證券，{daily["date"].min().date()} ~ {daily["date"].max().date()}')

# 交易日曆：用市場報酬指數 Y9997 有資料的日子
idx = daily[daily['stock_id'] == 'Y9997'].dropna(subset=['ret'])
calendar = pd.DatetimeIndex(sorted(idx['date'].unique()))
print(f'交易日曆（Y9997）：{len(calendar):,} 天')

# 只留候選公司的資料，轉成「日期 × 公司」的寬表。成交量為 0 的日子，報酬率視為缺值
ids = cand['stock_id'].unique()
sub = daily[daily['stock_id'].isin(ids)].drop_duplicates(['stock_id', 'date']).copy()
sub.loc[sub['vol'].fillna(0) <= 0, 'ret'] = np.nan
ret_wide = sub.pivot(index='date', columns='stock_id', values='ret').reindex(calendar)
cap_wide = sub.pivot(index='date', columns='stock_id', values='mktcap').reindex(calendar)

# ══ 2) 每則公告：事件日 t0、估計期有效天數、宣告前一個月底的市值 ══
def price_info(r):
    sid, d = r['stock_id'], r['ann_date']
    pos = int(calendar.searchsorted(d))
    if pos >= len(calendar):
        return pd.Series({'t0': pd.NaT, 'n_est': 0, 'mktcap': np.nan})
    if calendar[pos] == d and r['after_hours'] == '盤後':        # 交易日的盤後公告 → 順延一天
        pos += 1
    if pos >= len(calendar) or sid not in ret_wide.columns:
        return pd.Series({'t0': calendar[min(pos, len(calendar) - 1)], 'n_est': 0, 'mktcap': np.nan})
    lo, hi = max(0, pos - 150), pos - 31
    n_est = int(ret_wide[sid].iloc[lo:hi + 1].notna().sum()) if hi >= 0 else 0
    mpos = int(calendar.searchsorted(d.replace(day=1))) - 1       # 宣告前一個月的最後一個交易日
    cap = cap_wide[sid].iloc[max(0, mpos - 10):max(0, mpos) + 1].dropna()
    return pd.Series({'t0': calendar[pos], 'n_est': n_est, 'mktcap': cap.iloc[-1] if len(cap) else np.nan})

cand = cand.drop(columns=['t0', 'n_est', 'mktcap'], errors='ignore')
cand = pd.concat([cand, cand.apply(price_info, axis=1)], axis=1)
cand['has_price'] = cand['n_est'] >= 100
print('\n【公告當時有沒有足夠的股價資料】（估計期有效天數 ≥ 100）× 目前的上市別')
display(pd.crosstab(cand['listing'].fillna('（對不到）'), cand['has_price'].map({True: '有', False: '沒有'}), margins=True))

# ══ 3) 交易金額與相對規模（外幣用近似匯率，只供篩選參考）══
CUR = {'新台幣': 1, '新臺幣': 1, '台幣': 1, '臺幣': 1, 'NTD': 1, 'NT$': 1, 'TWD': 1, 'NT': 1,
       '美金': 29.5, '美元': 29.5, 'USD': 29.5, 'US$': 29.5, '人民幣': 4.3, 'RMB': 4.3, 'CNY': 4.3,
       '歐元': 33, 'EUR': 33, '日圓': 0.25, '日幣': 0.25, '日元': 0.25, 'JPY': 0.25, '港幣': 3.8, '港元': 3.8, 'HKD': 3.8,
       '新加坡幣': 21.5, '新幣': 21.5, '星幣': 21.5, 'SGD': 21.5, '英鎊': 37, 'GBP': 37, '韓元': 0.024, '韓圜': 0.024, 'KRW': 0.024,
       '泰銖': 0.9, 'THB': 0.9, '越南盾': 0.0012, 'VND': 0.0012, '馬幣': 6.9, '令吉': 6.9, 'MYR': 6.9, '印尼盾': 0.002, 'IDR': 0.002,
       '澳幣': 20.5, 'AUD': 20.5, '瑞士法郎': 31, 'CHF': 31, '加幣': 22, 'CAD': 22, '盧比': 0.38, 'INR': 0.38, '披索': 0.55, 'PHP': 0.55}
UNIT = {'億': 1e8, '千萬': 1e7, '仟萬': 1e7, '百萬': 1e6, '佰萬': 1e6, '萬': 1e4, '千': 1e3, '仟': 1e3, 'K': 1e3, 'M': 1e6, '': 1}
_cur = '|'.join(sorted((re.escape(k) for k in CUR), key=len, reverse=True))
_unit = '億|千萬|仟萬|百萬|佰萬|萬|千|仟|K|M'
AMT_RE = re.compile(r'(?P<c1>' + _cur + r')?\s*(?:約|約當|不超過|不高於|上限|最高|共計|計|為|:)*\s*(?P<c1b>' + _cur + r')?\s*'
                    r'(?P<n1>\d[\d,]*(?:\.\d+)?)\s*(?P<u1>' + _unit + r')?(?:(?P<n2>\d[\d,]*(?:\.\d+)?)\s*(?P<u2>' + _unit + r'))?'
                    r'\s*(?P<c2>' + _cur + r')?\s*(?:元|圓)?')
def parse_amount(text):
    """從文字裡找出交易金額，換算成新台幣百萬元（外幣用近似匯率，只供篩選參考）。回傳 (百萬元, 幣別)"""
    t = str(text).replace(' ', '')
    def value(m):
        cur = m.group('c1') or m.group('c1b') or m.group('c2')
        v = float(m.group('n1').replace(',', '')) * UNIT[m.group('u1') or '']
        if m.group('n2'): v += float(m.group('n2').replace(',', '')) * UNIT[m.group('u2') or '']
        return v, cur
    for key in ('交易總金額', '總金額', '交易總價格', '交易總額', '總價金', '總價款', '總對價', '對價總計', '投資金額', '交易金額', '價金總額', '股款', '共計', '合計'):
        i = t.find(key)
        if i < 0: continue
        for m in AMT_RE.finditer(t[i + len(key): i + len(key) + 80]):
            v, cur = value(m)
            fx = CUR.get(cur, 1) if cur else 1
            if v * fx < 1e5 or (v < 10000 and not cur): continue          # 太小的數字多半是股數、單價，跳過
            return round(v * fx / 1e6, 1), (cur or '未註明')
    best = None                                                    # 沒有關鍵字時：取有標明幣別的最大金額
    for m in AMT_RE.finditer(t[:400]):
        v, cur = value(m)
        if cur and v * CUR.get(cur, 1) >= 1e6 and (best is None or v * CUR[cur] > best[0]):
            best = (v * CUR[cur], cur)
    if best: return round(best[0] / 1e6, 1), best[1]
    return None, ''

txt = lambda v: v if isinstance(v, str) else ''
amt = [parse_amount(txt(a) if txt(a) else '') for a in cand['amount_hint']]
cand['amt_guess'] = [a[0] for a in amt]
cand['amt_cur']   = [a[1] for a in amt]
cand['rel_size_guess'] = (cand['amt_guess'] / cand['mktcap']).round(4)

# ══ 4) 補強自動標記 ══
# 4a 公告當時是不是興櫃公司：興櫃公司用的是另一套規章，款的編號不同（併購 = 第 9 款、取得股權 = 第 17 款、其他 = 第 43、44 款）
cl = pd.to_numeric(cand['clause'], errors='coerce')
cand['emerging'] = (cand['template'].isin(['併購', '股份交換']) & (cl == 9)) | \
                   (cand['template'].isin(['取得有價證券', '買賣有價證券']) & (cl == 17)) | cl.isin([43, 44])

# 4b 取得股權的表格：交易「前」的持股比例 = 交易後持股 ×（1 − 本次買進股數 ÷ 交易後累積股數）
def section(content, start, stop):
    c = nfkc(content).replace('\n', '')
    m = re.search(start, c)
    if not m: return ''
    rest = c[m.end():]
    e = re.search(stop, rest)
    return rest[:e.start()] if e else rest[:400]
def first_shares(seg):
    m = re.search(r'(\d[\d,]*(?:\.\d+)?)\s*(仟|千|萬|百萬|億)?\s*股(?!份有限|權|東|利|本)', seg)
    if not m: return None
    return float(m.group(1).replace(',', '')) * {'仟': 1e3, '千': 1e3, '萬': 1e4, '百萬': 1e6, '億': 1e8}.get(m.group(2), 1)
def pct_before(r):
    if r['template'] not in ('取得有價證券', '買賣有價證券') or pd.isna(r['pct_after']): return np.nan
    bought = first_shares(section(r['content'], r'交易數量、每單位價格及交易總金額[::]?', r'4\.交易相對人'))
    total  = first_shares(section(r['content'], r'累積持有本交易證券[^::]{0,60}[::]', r'1[34]\.'))
    if not bought or not total or bought > total * 1.001: return np.nan
    return round(r['pct_after'] * (1 - bought / total), 2)
cand['pct_before'] = cand.apply(pct_before, axis=1)

# 4c 集團內部重組：分成「證據明確」和「只知道是關係人」兩種。前者建議排除，後者要人工確認
MAJ = r'(?<![\d.])(?:5[1-9]|[6-9]\d|100)(?:\.\d+)?%'                 # 51%～100%（前面不能接數字或小數點，避免把 5.80% 看成 80%）
IN_SUBJ = re.compile(r'簡易合併|簡易股份轉換|子公司間|[二兩2]家(?:子|孫)公司|(?:投資|持股|組織|集團)架構|架構(?:調整|重整|重組)|組織重整|組織調整|組織重組|集團內'
                     r'|合併.{0,10}子公司|與.{0,12}子公司.{0,40}(?:合併|股份轉換)|取得.{0,10}子公司|剩餘股權|其餘股權|少數股權|剩餘.{0,6}股份')
IN_REL  = re.compile(r'[均皆同]為.{0,50}(?:子公司|孫公司)|母公司[均皆同]為|母子公司|子公司間|組織(?:架構)?(?:重整|調整|重組)|集團內|簡易合併|企業併購法第(?:19|十九)條'
                     r'|本公司[^,。;公司]{0,14}(?:子公司|孫公司)'                                    # 本公司（100% 持有）之子公司
                     r'|本公司[^,。;]{0,12}持[有股份]{1,3}[^,。;%]{0,12}?' + MAJ)                   # 本公司轉投資持股達 53.84%
PARENT  = re.compile(r'本公司之?(?:最終)?母公司|同一母公司|持有本公司|對本公司持股|投資本公司|控制本公司')            # 對方是本公司的母公司或大股東 → 不算內部重組
PCT_RE = re.compile(r'(\d{1,3}(?:\.\d+)?)\s*%')
def flags2(r):
    f = [x for x in txt(r['auto_flag']).split('、') if x]
    s, tgt, rel, dtp = txt(r['subject']).replace('取得或處分資產', ''), txt(r['target_raw']), txt(r['relation']), txt(r['deal_type'])
    cp = txt(r['counterparty'])
    is_sec = r['template'] in ('取得有價證券', '買賣有價證券')
    if '含處分' in f and '處分' not in s: f.remove('含處分')
    name = re.sub(r'[-*]|KY', '', txt(r['firm_name']))
    def add(x):
        if x not in f: f.append(x)
    # 集團內部重組：重新判斷證據強弱
    hinted = '集團內部重組?' in f or '子公司' in rel
    if '集團內部重組?' in f: f.remove('集團內部重組?')
    by_rel = bool(IN_REL.search(rel + ' ' + cp)) and not PARENT.search(rel + ' ' + cp)
    tkey = re.sub(r'[^一-鿿]', '', tgt)[:4]                         # 標的名稱的前四個字
    if is_sec and r['pct_before'] == 0 and len(tkey) == 4 and tkey in cp:   # 原本沒有持股、對方欄位寫的就是標的本身 → 是新取得，不算內部重組
        by_rel = False
    strong = bool(IN_SUBJ.search(s)) or by_rel or (is_sec and bool(re.search(r'發行新股|現金增資', cp)))
    if strong:                                                             add('集團內部重組?')
    elif hinted:                                                           add('關係人交易' if r['related'] == '是' else '集團內?待確認')
    if pd.notna(r['pct_before']) and r['pct_before'] > 50:                 add('原已持股過半')
    if r['emerging']:                                                      add('公告時為興櫃?')
    if re.search(r'累積取得|一年內', s):                                   add('陸續買進')
    if re.search(r'讓售|出售|處分|營業讓與', s) or ('讓與' in dtp and '受讓' not in dtp): add('處分或讓與')
    if re.search(r'取得.{0,30}現金增資|現金增資股|新設子公司|設立子公司|認購.{0,20}增資', s): add('增資或新設')
    if re.search(r'新聞稿|檢調|知悉|結果|票決|提案|解除|取消|撤回|停止執行|增補|價金調整|調整後|已完成|實際執行|生效|發放|質權人|辭任|時程', s):
        add('後續公告?')
    if re.search(r'新設.{0,12}控股|共同.{0,6}股份轉換', s + tgt):           add('共同新設控股公司')
    if r['template'] == '股份交換' and pd.notna(r['pct_after']) and r['pct_after'] <= 50: add('少數股權?')
    if '公開收購' in dtp + s:                                              # 公開收購：預定收購的比例不到一半，不一定取得控制權
        p = [float(x) for x in PCT_RE.findall(tgt + s) if float(x) <= 100]
        if p and max(p) <= 50:                                             add('公開收購比例≤50%?')
    me = r'(?:本公司|' + re.escape(name) + r')' if name else r'本公司'
    if re.search(r'(?:被收購公司|讓與方|消滅公司|轉換公司)[:：][^:：]{0,40}' + me, tgt) or \
       re.search(me + r'[^,。;]{0,30}為消滅公司', tgt):
        add('本公司是被併方?')
    if pd.notna(r['rel_size_guess']) and r['rel_size_guess'] < 0.005:      add('規模<1%?')
    return '、'.join(f)

cand['flag2'] = cand.apply(flags2, axis=1)
STRONG2 = r'金融業|分割|集團內部重組|原已持股過半|本公司是被併方|非普通股|增資或新設|含處分|少數股權|陸續買進|處分或讓與|規模<1%'
cand['prio2'] = np.where(cand['flag2'] == '', 'A', np.where(cand['flag2'].str.contains(STRONG2), 'C', 'B'))

# ══ 5) 把同一宗交易的多則公告歸成一組 ══
STOP_RE = re.compile('股份有限公司|有限公司|\\(股\\)公司|\\(股\\)|公司|本公司|子公司|孫公司|重要|代|公告|董事會|董事長|決議|通過|擬|以|與|及|之|的|向|等|案|事宜|相關|有關|'
    '取得|收購|公開|合併|併購|股份|轉換|交換|受讓|讓與|股權|普通股|股票|全部|全數|已發行|現金|對價|方式|增資|發行|新股|進行|簽訂|簽署|契約|協議|'
    '補充|更正|說明|訂定|基準日|變更|調整|核准|申報|生效|完成|終止|董事|股東會|臨時會|常會|授權|辦理|作為|持有|'
    '公開收購人|條件|期間|延長|屆滿|說明書|意見書|修正|再次|百分之百|簡易|吸收|集團|架構|間接|直接|透過|新設|將|為|其|所|於|並|暨|或|至|由|因|依|據|規定|處理|準則|資產|處分')
LATIN_STOP = {'ltd', 'inc', 'corp', 'corporation', 'limited', 'holding', 'holdings', 'company', 'gmbh', 'international', 'technology',
              'group', 'global', 'investment', 'investments', 'cayman', 'samoa', 'electronics', 'enterprise', 'co.', 'pte', 'llc'}
def name_keys(text):
    t = STOP_RE.sub(' ', re.sub(r'\d+(?:\.\d+)?%?', ' ', text))
    keys = {w.lower().strip('.') for w in re.findall(r'[A-Za-z][A-Za-z&\.\-]{3,}', t)} - LATIN_STOP
    for seg in re.findall(r'[一-鿿]{2,}', t):
        keys |= {seg[i:i + 2] for i in range(len(seg) - 1)}
    return keys

cand = cand.sort_values(['stock_id', 'ann_date', 'ann_time']).reset_index(drop=True)
tw = lambda v: txt(v).replace('臺', '台')
texts = [tw(s) + ' ' + tw(t) for s, t in zip(cand['subject'], cand['target_raw'])]
keys = [name_keys(t) for t in texts]
OWN = re.compile(r'([一-鿿]{2,14}?)(?:股份有限公司|\(股\)公司)?[\(（][^\)）]{0,14}本公司')
bigrams = lambda nm: {nm[k:k + 2] for k in range(len(nm) - 1)}
for sid, g in cand.groupby('stock_id', sort=False):
    ix = g.index.tolist()
    # 公司自己的名字不拿來比對：簡稱、可能的全名（例如 台泥 → 台灣水泥）、全文裡「○○公司(以下簡稱本公司)」的名字
    short = re.sub(r'[^一-鿿]', '', tw(g['firm_name'].iloc[0]))
    own = bigrams(short)
    full = re.compile('[一-鿿]{0,2}'.join(short)) if len(short) >= 2 else None
    cnt = {}
    for i in ix:
        if full:
            for nm in full.findall(texts[i]): own |= bigrams(nm)
        for nm in OWN.findall(texts[i]):     own |= bigrams(nm)
        for k in keys[i]:
            if k[0].isascii(): cnt[k] = cnt.get(k, 0) + 1
    if len(ix) >= 5:                                   # 英文名字：這家公司四成以上的公告都出現的字，多半是集團自己的名字
        own |= {k for k, c in cnt.items() if c >= 0.4 * len(ix)}
    for i in ix: keys[i] = keys[i] - own
# 只用「有辨識力」的字比對：出現在很多家公司公告裡的字（地名、行業名、套語）不算
firm_df = {}
for sid, g in cand.groupby('stock_id', sort=False):
    for k in set().union(*[keys[i] for i in g.index]): firm_df[k] = firm_df.get(k, 0) + 1
keys = [{k for k in ks if firm_df[k] <= 4} for ks in keys]
parent = list(range(len(cand)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]; i = parent[i]
    return i
dates = cand['ann_date'].tolist()
# 標的欄位去掉空白和標點後一字不差，也算同一宗（太短或只寫「本公司」「普通股」的不算）
tnorm = [re.sub(r'[\W_]+', '', tw(t)).lower() for t in cand['target_raw']]
tnorm = [t if len(t) >= 4 and not re.match(r'本公司|存續公司|普通股|不適用', t) else '' for t in tnorm]
for sid, g in cand.groupby('stock_id', sort=False):
    ix = g.index.tolist()
    for a in range(len(ix)):
        for b in range(a + 1, len(ix)):
            i, j = ix[a], ix[b]
            if (dates[j] - dates[i]).days > 365: break
            both = keys[i] & keys[j]                  # 共同的字要有兩個以上，或是只有這家公司用過的字
            if len(both) >= 2 or any(firm_df[k] == 1 for k in both) or (tnorm[i] and tnorm[i] == tnorm[j]):
                parent[find(j)] = find(i)
cand['deal_gid'] = [find(i) for i in range(len(cand))]

# ══ 6) 以「宗」為單位整理 ══
ILLEGAL = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f]')
clean = lambda v: ILLEGAL.sub('', v) if isinstance(v, str) else v
CROSS = (r'[A-Za-z]{4,}|株式會社|日本|美國|香港|越南|泰國|德國|法國|英國|韓國|印尼|印度|馬來西亞|新加坡|菲律賓|荷蘭|義大利|澳洲|加拿大|墨西哥'
         r'|大陸|中國|上海|北京|深圳|廣州|東莞|昆山|蘇州|無錫|天津|重慶|成都|廈門|杭州|南京|寧波|青島|武漢|常熟|常州|惠州|珠海|佛山|江蘇|浙江|廣東|福建|四川')
BIZ = re.compile(r'(?:所營業務|主要營業項目|主要業務|營業項目|主要從事|主要產品|主要經營)[^。\n]{0,90}')
deals = []
for gid, g in cand.groupby('deal_gid', sort=False):
    g = g.sort_values(['ann_date', 'ann_time'])
    first, forms = g.iloc[0], g[g['template'] != '其他']
    main = forms.iloc[0] if len(forms) else first                  # 以最早的制式表格公告為主要依據
    content = txt(main['content'])
    pay = txt(main['payment_guess'])
    if not pay and main['template'] == '併購':                      # 舊版表格沒有對價欄位，改從全文和主旨判斷
        z = txt(main['ratio']) + txt(main['subject']) + content[:3000]
        c, s_ = bool(re.search(r'現金為對價|以現金|現金對價|每股現金', z)), bool(re.search(r'發行新股|增資發行|換發', z))
        pay = 'mixed?' if c and s_ else 'cash?' if c else 'stock?' if s_ else ''
    biz = BIZ.search(content.replace('\n', ''))
    note = []
    later = forms[~forms['flag2'].str.contains('後續公告')].index[1:]
    if any(keys[i] and not (keys[i] & keys[main.name]) for i in later): note.append('組內可能有不只一宗交易，請看同組公告')
    if len(forms) and first.name != main.name:                      note.append('最早的公告不是制式表格，請確認首次公告日')
    deals.append({
        'deal_gid': gid, 'tier': '第一層' if len(forms) else '第二層', 'prio': main['prio2'], 'flags': main['flag2'],
        'all_flags': '、'.join(sorted({x for fl in g['flag2'] for x in fl.split('、') if x})),
        'stock_id': first['stock_id'], 'firm_name': first['firm_name'], 'acq_industry': txt(first['tej_ind']), 'listing': txt(first['listing']),
        'first_date': first['ann_date'], 'first_time': first['ann_time'], 'after_hours': first['after_hours'], 't0': first['t0'],
        'n_est': first['n_est'], 'mktcap': first['mktcap'], 'has_price': first['has_price'],
        'template': main['template'], 'main_date': main['ann_date'], 'subject': main['subject'], 'deal_type': main['deal_type'],
        'target_raw': main['target_raw'], 'counterparty': main['counterparty'], 'related': main['related'], 'relation': main['relation'],
        'consideration': main['consideration'], 'payment_guess': pay, 'ratio': main['ratio'], 'pct_after': main['pct_after'],
        'amount_hint': main['amount_hint'], 'amt_guess': main['amt_guess'], 'amt_cur': main['amt_cur'], 'rel_size_guess': main['rel_size_guess'],
        'check': '；'.join(note), 'pct_before': main['pct_before'],
        'first_subject': first['subject'], 'first_content': txt(first['content'])[:30000] if first.name != main.name else '',
        'cross_guess': '1?' if re.search(CROSS, txt(main['target_raw']) + txt(main['subject'])) else '0?',
        'biz_hint': biz.group(0) if biz else '', 'n_ann': len(g),
        'members': '\n'.join(f"{d.date()} [{t}] {s[:60]}" for d, t, s in zip(g['ann_date'], g['template'], g['subject'])),
        'content': content[:30000]})
deals = pd.DataFrame(deals)
deals['in_period'] = deals['first_date'] >= '2019-01-01'
# 同一家公司前後一年內的其他候選交易（幫忙判斷：是不是同一宗被拆開了、60 個交易日內有沒有另一宗）
near = []
for sid, g in deals.groupby('stock_id', sort=False):
    for i, r in g.iterrows():
        o = g[(g.index != i) & ((g['first_date'] - r['first_date']).abs().dt.days <= 365)].sort_values('first_date')
        near.append((i, '\n'.join(f"{d.date()} [{p}] {s[:45]}" for d, p, s in zip(o['first_date'], o['prio'], o['subject']))))
deals['nearby'] = pd.Series(dict(near))

# ══ 7) 統計 ══
print(f'\n候選公告 {len(cand):,} 則 → 歸組後 {len(deals):,} 宗')
print(f'  首次公告在 2019 年之前（不在樣本期間）：{int((~deals["in_period"]).sum()):,} 宗')
d1 = deals[deals['in_period']]
print(f'  公告當時沒有足夠的股價資料：{int((~d1["has_price"]).sum()):,} 宗')
rev = d1[d1['has_price']].copy()
print(f'  → 可以審閱的：{len(rev):,} 宗')
print('\n【層級 × 優先順序】第一層 = 至少有一則制式表格公告；第二層 = 只有自由格式公告')
display(pd.crosstab(rev['tier'], rev['prio'], margins=True))
r1 = rev[rev['tier'] == '第一層']
print('\n【第一層：表格類型 × 優先順序】'); display(pd.crosstab(r1['template'], r1['prio'], margins=True))
print('\n【第一層：年份 × 優先順序】');     display(pd.crosstab(r1['first_date'].dt.year, r1['prio'], margins=True))
print('\n【第一層 A、B 的自動標記】'); print(r1[r1['prio'] != 'C']['flags'].replace('', '（無）').value_counts().head(12).to_string())
print('\n【金額解析】第一層 A、B 中，有抓到金額的：', int(r1[r1['prio'] != 'C']['amt_guess'].notna().sum()), '宗；幣別：',
      r1[r1['prio'] != 'C']['amt_cur'].replace('', '（無）').value_counts().head(8).to_dict())
print('【相對規模（估計值）的分布】'); print(r1[r1['prio'] != 'C']['rel_size_guess'].describe().round(4).to_string())

# ══ 8) 存檔 ══
cand.to_pickle(f'{WORK_DIR}/candidates_v3.pkl')
deals.to_pickle(f'{WORK_DIR}/deals_v3.pkl')
order = {'A': 0, 'B': 1, 'C': 2}
out = rev.assign(_o=rev['prio'].map(order)).sort_values(['tier', '_o', 'stock_id', 'first_date']).drop(columns=['_o', 'in_period', 'has_price'])
out_path = f'{WORK_DIR}/deals_v3.xlsx'
with pd.ExcelWriter(out_path) as xw:
    for tier in ['第一層', '第二層']:
        o = out[out['tier'] == tier].copy()
        for c in ['content', 'first_content']: o[c] = o[c].str[:8000]
        for c in o.columns: o[c] = o[c].map(clean)
        o.to_excel(xw, sheet_name=tier, index=False)
print(f'\n已存出：{out_path}（存在 Drive 的 work 資料夾，不會自動下載）')

## 第 5 格｜產生審閱用的檔案

程式只能做到「挑出候選、預填線索」，是否符合樣本條件要由人逐宗判斷。這一格產生三個檔案（都在 Drive 的 `ma_event_study/work/`）：

| 檔案 | 用途 |
|---|---|
| `review_v1_filled.xlsx` | **你填寫的審閱表**。三個分頁：「審閱」（第一層的 A 和 B）、「C抽查」、「說明」 |
| `review_v1_blank.xlsx` | 空白備份，不要動 |
| `review_v1_reader.html` | **全文閱讀頁**。每宗交易一張卡片：上面是程式整理的重點，下面是公告全文。用編號和審閱表對照 |

**C 抽查**：程式建議排除的（C 級）不逐宗審閱，改成隨機抽 60 宗，由人判斷程式排除得對不對，用來估計程式漏掉了多少真正的併購。亂數種子固定，每次抽到的都一樣。

**保護機制**：`review_v1_filled.xlsx` 已經存在時，這一格不會覆寫它，你填過的內容不會不見。

In [ ]:
import html as _html, shutil
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter

TAG = 'v1'                                   # 審閱表的版本
N_AUDIT = 60                                 # C 級抽查的宗數
r1 = deals[deals['in_period'] & deals['has_price'] & (deals['tier'] == '第一層')].copy()
r1['key'] = r1['stock_id'] + '_' + r1['first_date'].dt.strftime('%Y%m%d') + '_' + r1['main_date'].dt.strftime('%Y%m%d') + '_' + r1['deal_gid'].astype(str)

# ══ 1) 要審閱的：第一層的 A 和 B，照公司、日期排序（同一家公司的交易排在一起，方便比對）══
todo = r1[r1['prio'] != 'C'].sort_values(['stock_id', 'first_date', 'main_date']).reset_index(drop=True)
todo.insert(0, 'no', [str(i) for i in range(1, len(todo) + 1)])
# ══ 2) 抽查用的：從第一層的 C 隨機抽 60 宗（固定亂數種子，每次抽到的都一樣）══
c_all = r1[r1['prio'] == 'C']
audit = c_all.sample(n=min(N_AUDIT, len(c_all)), random_state=20261003).sort_values(['stock_id', 'first_date']).reset_index(drop=True)
audit.insert(0, 'no', [f'C{i}' for i in range(1, len(audit) + 1)])

# ══ 3) 審閱表（Excel）══
INPUTS = ['納入', '排除原因', '標的公司', '支付方式', '跨國', '產業關係', '標的上市櫃', '交易金額_百萬元', '發行新股_千股', '首次公告日_更正', '備註']
HINTS = {'stock_id': '代號', 'acq_industry': '主併產業', 'after_hours': '盤中盤後', 't0': '事件日t0', 'prio': '優先', 'flags': '自動標記', 'check': '提醒',
         'template': '表格', 'deal_type': '併購種類', 'target_raw': '標的(原文)', 'counterparty': '交易相對人', 'related': '關係人',
         'pct_before': '交易前持股%', 'pct_after': '交易後持股%', 'payment_guess': '支付(猜)', 'consideration': '對價說明', 'cross_guess': '跨國(猜)',
         'biz_hint': '標的業務(線索)', 'amount_hint': '金額線索', 'amt_guess': '金額(猜)_百萬元', 'amt_cur': '幣別', 'mktcap': '市值_百萬元',
         'rel_size_guess': '相對規模(猜)', 'n_ann': '公告數', 'members': '同組公告', 'nearby': '同公司其他候選', 'key': 'key'}
def sheet_df(df, inputs):
    o = pd.DataFrame({'編號': df['no'], '公司': df['stock_id'] + ' ' + df['firm_name'], '首次公告日': df['first_date'].dt.date, '主旨': df['subject']})
    for c in inputs: o[c] = ''
    for k, v in HINTS.items(): o[v] = df[k].values
    o['事件日t0'] = pd.to_datetime(o['事件日t0']).dt.date
    for c in o.columns: o[c] = o[c].map(clean)
    return o
GUIDE = [
 ('怎麼用', '一列是一宗交易。左邊黃色欄位由你填，右邊灰色欄位是程式預填的線索（有「猜」字的不保證正確）。全文請看 review_%s_reader.html，用編號對照。' % TAG),
 ('納入', '1 = 符合樣本條件；0 = 不符合。0 的話只要再填「排除原因」，其他欄位不用填。'),
 ('排除原因', '少數股權（交易後持股 ≤ 50%，或公開收購後仍未過半）／集團內部（交易前已經控制標的）／資產購買（買的是土地、設備，不是公司或事業）／'
              '非首次公告（同一宗交易更早已公告過）／被併方（本公司是被收購的一方）／興櫃（公告當時不是上市櫃）／規模小於1%／60日內重複（同公司 60 個交易日內的第二宗）／其他（在備註寫原因）'),
 ('標的公司', '被併購的公司名稱（寫簡稱即可）。'),
 ('支付方式', 'cash = 全部現金；stock = 全部發行新股或換股；mixed = 現金加股票。'),
 ('跨國', '1 = 標的公司的主要營運地在台灣以外（含中國大陸）；0 = 在台灣。看營運地，不看註冊地（註冊在開曼、營運在台灣的算 0）。'),
 ('產業關係', '同業 = 標的和主併公司做同一類產品或服務；上下游 = 標的是主併公司的供應商或客戶；無關 = 其他。依公告裡的「併購目的」「標的業務」判斷，在備註寫一句理由。'),
 ('標的上市櫃', '1 = 標的是上市、上櫃或興櫃公司（有公開股價）；0 = 未上市。'),
 ('交易金額_百萬元', '這宗交易的總對價，換算成新台幣百萬元。外幣先填原幣金額在備註（例如 USD 50M），之後由程式用公告日匯率換算。公開收購 = 每股價格 × 預定最高收購股數。'),
 ('發行新股_千股', '只有 stock 或 mixed 才填：主併公司為這宗交易發行的新股股數（千股）。'),
 ('首次公告日_更正', '如果你發現這宗交易更早就公告過（看「同組公告」「同公司其他候選」），填更早的日期（西元年/月/日）。沒有就留白。'),
 ('備註', '判斷的理由、不確定的地方。之後回頭檢查、寫方法說明時會用到。'),
 ('C抽查', '這一頁是程式建議排除的交易，隨機抽 %d 宗。請判斷「程式排除得對不對」：應該納入的填 1，排除得對的填 0。用來估計程式漏掉多少。' % N_AUDIT),
]
def style(ws, n_in, lists):
    yellow, grey = PatternFill('solid', fgColor='FFF2CC'), PatternFill('solid', fgColor='EDEDED')
    for j, cell in enumerate(ws[1], start=1):
        cell.font = Font(bold=True); cell.alignment = Alignment(wrap_text=True, vertical='top')
        cell.fill = yellow if 5 <= j < 5 + n_in else grey
    ws.freeze_panes = 'E2'; ws.row_dimensions[1].height = 32
    width = {'編號': 6, '公司': 15, '首次公告日': 12, '主旨': 46, '排除原因': 14, '標的公司': 18, '備註': 30, '標的(原文)': 36, '交易相對人': 28,
             '對價說明': 28, '標的業務(線索)': 36, '金額線索': 30, '同組公告': 50, '同公司其他候選': 50, '自動標記': 22, '提醒': 22}
    for j, cell in enumerate(ws[1], start=1):
        ws.column_dimensions[get_column_letter(j)].width = width.get(cell.value, 12)
    last = ws.max_row
    for name, opts in lists.items():
        col = [c.value for c in ws[1]].index(name) + 1
        dv = DataValidation(type='list', formula1='"' + ','.join(opts) + '"', allow_blank=True)
        ws.add_data_validation(dv); dv.add(f'{get_column_letter(col)}2:{get_column_letter(col)}{last}')
LISTS = {'納入': ['1', '0'], '排除原因': ['少數股權', '集團內部', '資產購買', '非首次公告', '被併方', '興櫃', '規模小於1%', '60日內重複', '其他'],
         '支付方式': ['cash', 'stock', 'mixed'], '跨國': ['1', '0'], '產業關係': ['同業', '上下游', '無關'], '標的上市櫃': ['1', '0']}
blank = f'{WORK_DIR}/review_{TAG}_blank.xlsx'
with pd.ExcelWriter(blank, engine='openpyxl') as xw:
    sheet_df(todo, INPUTS).to_excel(xw, sheet_name='審閱', index=False)
    sheet_df(audit, ['應該納入', '備註']).to_excel(xw, sheet_name='C抽查', index=False)
    pd.DataFrame(GUIDE, columns=['欄位', '說明']).to_excel(xw, sheet_name='說明', index=False)
    style(xw.sheets['審閱'], len(INPUTS), LISTS)
    style(xw.sheets['C抽查'], 2, {'應該納入': ['1', '0']})
    xw.sheets['說明'].column_dimensions['A'].width = 18; xw.sheets['說明'].column_dimensions['B'].width = 120
    for row in xw.sheets['說明'].iter_rows(min_row=2):
        for cell in row: cell.alignment = Alignment(wrap_text=True, vertical='top')
filled = f'{WORK_DIR}/review_{TAG}_filled.xlsx'
if os.path.exists(filled):
    print(f'⚠️ 已經有 review_{TAG}_filled.xlsx（你填寫中的檔案），這次沒有覆寫它。')
else:
    shutil.copy(blank, filled)
    print(f'已建立你要填寫的檔案：review_{TAG}_filled.xlsx')

# ══ 4) 全文閱讀頁（HTML）：一宗一張卡片，上面是程式整理的重點，下面是公告全文 ══
esc = lambda v: _html.escape('' if v is None or (isinstance(v, float) and pd.isna(v)) else str(v))
def card(r):
    rows = [('自動標記', r['flags'] or '（無）'), ('提醒', r['check']), ('表格 / 併購種類', f"{r['template']} / {txt(r['deal_type'])}"),
            ('標的（原文）', r['target_raw']), ('交易相對人', r['counterparty']), ('關係人', f"{txt(r['related'])}　{txt(r['relation'])}"),
            ('持股：交易前 → 後', f"{'' if pd.isna(r['pct_before']) else r['pct_before']} → {'' if pd.isna(r['pct_after']) else r['pct_after']} %"),
            ('支付（猜）/ 對價說明', f"{txt(r['payment_guess'])}　{txt(r['consideration'])}　{txt(r['ratio'])}"),
            ('金額線索', r['amount_hint']),
            ('金額（猜）', '程式沒抓到金額，請從全文找' if pd.isna(r['amt_guess']) else
                          f"約 {r['amt_guess']:,.1f} 百萬元（{txt(r['amt_cur'])}）÷ 市值 {r['mktcap']:,.0f} 百萬元 = 相對規模約 {r['rel_size_guess']:.1%}"),
            ('主併公司市值', '' if pd.isna(r['mktcap']) else f"{r['mktcap']:,.0f} 百萬元（宣告前一個月底）"),
            ('主併產業 / 標的業務', f"{txt(r['acq_industry'])}　/　{txt(r['biz_hint'])}"), ('同組公告', r['members']), ('同公司其他候選', r['nearby'])]
    tb = ''.join(f'<tr><th>{esc(k)}</th><td>{esc(v)}</td></tr>' for k, v in rows if txt(v).strip())
    first = (f"<details><summary>最早的公告（{esc(r['first_date'].date())}）全文：{esc(r['first_subject'])}</summary><pre>{esc(r['first_content'])}</pre></details>"
             if txt(r['first_content']) else '')
    return (f"<section id='d{esc(r['no'])}'><h2><span class='no'>{esc(r['no'])}</span> {esc(r['stock_id'])} {esc(r['firm_name'])}　"
            f"<small>首次公告 {esc(r['first_date'].date())} {esc(r['first_time'])}（{esc(r['after_hours'])}）　[{esc(r['prio'])}]</small></h2>"
            f"<p class='subj'>{esc(r['subject'])}</p><table>{tb}</table>{first}"
            f"<details open><summary>制式表格公告全文（{esc(r['main_date'].date())}）</summary><pre>{esc(clean(r['content']))}</pre></details></section>")
CSS = ('body{font-family:-apple-system,"PingFang TC","Microsoft JhengHei",sans-serif;margin:0;background:#f4f4f2;color:#222;line-height:1.6}'
       'header{position:sticky;top:0;background:#1f3a5f;color:#fff;padding:10px 16px;display:flex;gap:10px;align-items:center;z-index:9;flex-wrap:wrap}'
       'header input{width:80px;padding:6px;font-size:16px}header button{padding:6px 12px;font-size:15px}'
       'section{background:#fff;margin:16px auto;padding:16px 20px;max-width:980px;border-radius:8px;box-shadow:0 1px 3px rgba(0,0,0,.12)}'
       'h2{margin:0 0 4px;font-size:20px}h2 small{font-weight:400;font-size:14px;color:#555}.no{background:#1f3a5f;color:#fff;border-radius:4px;padding:2px 9px;margin-right:6px}'
       '.subj{font-weight:600;margin:4px 0 10px}table{border-collapse:collapse;width:100%;font-size:14px;margin-bottom:10px}'
       'th{text-align:left;width:150px;vertical-align:top;color:#555;font-weight:500;padding:3px 8px 3px 0}td{padding:3px 0;white-space:pre-wrap;word-break:break-word}'
       'tr{border-bottom:1px solid #eee}pre{white-space:pre-wrap;word-break:break-word;font-family:inherit;font-size:14px;background:#fafaf7;padding:12px;border-radius:6px}'
       'summary{cursor:pointer;color:#1f3a5f;font-weight:600;margin:6px 0}')
JS = ("function go(d){var v=document.getElementById('n').value.trim().toUpperCase();if(d){var m=v.match(/^(C?)(\\d+)$/);if(m)v=m[1]+(parseInt(m[2])+d);}"
      "var e=document.getElementById('d'+v);if(e){document.getElementById('n').value=v;e.scrollIntoView();window.scrollBy(0,-60);}}")
page = (f"<!doctype html><html lang='zh-Hant'><head><meta charset='utf-8'><meta name='viewport' content='width=device-width,initial-scale=1'>"
        f"<title>併購候選公告閱讀頁 {TAG}</title><style>{CSS}</style></head><body><header><b>候選公告閱讀頁 {TAG}</b>"
        f"<span>編號</span><input id='n' value='1' onkeydown=\"if(event.key==='Enter')go(0)\"><button onclick='go(0)'>跳到</button>"
        f"<button onclick='go(-1)'>上一宗</button><button onclick='go(1)'>下一宗</button>"
        f"<span>審閱 1～{len(todo)}；C 抽查 C1～C{len(audit)}</span></header>"
        + ''.join(card(r) for _, r in todo.iterrows()) + ''.join(card(r) for _, r in audit.iterrows())
        + f"<script>{JS}</script></body></html>")
reader = f'{WORK_DIR}/review_{TAG}_reader.html'
with open(reader, 'w', encoding='utf-8') as fh: fh.write(page)

# ══ 5) 統計 ══
print(f'\n要審閱的交易：{len(todo)} 宗（A {int((todo["prio"] == "A").sum())}、B {int((todo["prio"] == "B").sum())}），來自 {todo["stock_id"].nunique()} 家公司')
print(f'C 級共 {len(c_all)} 宗，隨機抽 {len(audit)} 宗做抽查')
print('\n【要審閱的交易：年份 × 表格類型】'); display(pd.crosstab(todo['first_date'].dt.year, todo['template'], margins=True))
print('\n【要審閱的交易：自動標記】'); print(todo['flags'].replace('', '（無）').value_counts().head(12).to_string())
print(f'\n檔案都在 Drive 的 ma_event_study/work 資料夾：')
print(f'  review_{TAG}_filled.xlsx  ← 你填寫的檔案（在 Drive 用 Google 試算表開啟）')
print(f'  review_{TAG}_blank.xlsx   ← 空白備份，不要動')
print(f'  review_{TAG}_reader.html  ← 全文閱讀頁（會自動下載，用瀏覽器開啟）')
from google.colab import files as colab_files
colab_files.download(reader)